# Demographics imputation

In [ ]:
import pandas as pd
from collections import Counter

#from sklearn.linear_model import LogisticRegression
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.metrics import classification_report

from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, balanced_accuracy_score

#from sklearn import preprocessing 

from tableone import TableOne
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import KNNImputer

### Load Data

In [ ]:
# Load Data
merged_df = pd.read_csv('patient_profile_analysis.csv')
# Replace NaN values 
merged_df['opioid'] = merged_df['opioid'].fillna(0)
merged_df['primaryinsurance'] = merged_df['primaryinsurance'].fillna('unknown')

# Define variables
missing_vars = ['smokingstatus', 'socialsupport', 'raceethnicity', 'primaryinsurance']  #'religion', 
predictors = ['sex', 'age_firstLBP', 'startdatekey_firstLBP', 'preferredlanguage', 'primaryinsurance', 
              'smokingstatus', 'socialsupport', 'raceethnicity']

df_impute = merged_df[set(missing_vars + predictors)]
df_impute

### Characterize Data

In [ ]:
# --- DESCRIBE DATA ---
df_info = merged_df[set(missing_vars + ['opioid', 'startdatekey_firstLBP'])]
df_info = df_info[df_info['startdatekey_firstLBP']>20120000]

df_summary = TableOne(df_info, 
                      groupby=f'opioid', 
                      categorical=missing_vars,
                      pval=True)
df_summary

In [ ]:
# --- DESCRIBE VALUES ---
for col_name in set(missing_vars + predictors): #
    print('--new line')
    print(df_impute.value_counts(col_name))
#     print(df_impute[df_impute['opioid']==1].value_counts(col_name))
#     print(df_impute[df_impute['opioid']==0].value_counts(col_name))
    
df_impute

In [ ]:
# --- DESCRIBE UNKNOWNS ---
# Count the number of 'unknown' in each row
unknown_counts = df_impute.apply(lambda row: (row == 'unknown').sum(), axis=1)

# Generate summary
summary = Counter(unknown_counts)

# Convert summary to a readable format
df_summary = pd.DataFrame.from_dict(summary, orient='index', columns=['Row Count']).sort_index()
df_summary.index.name = 'Number of Unknowns'

print(df_summary)

# Function to count other columns with 'unknown' when column 'a' is 'unknown'
def count_unknowns_when_col_unknown(row, col):
    if row[col] == 'unknown':
        return (row.drop(col) == 'unknown').sum()  # Count 'unknown' in other columns
    return 0  # Return 0 if column 'a' is not 'unknown'

# Apply the function to each row
for col in missing_vars:
    total_other_unknowns = df_impute.apply(lambda row: count_unknowns_when_col_unknown(row, col), axis=1)
    print(f"When column '{col}' is 'unknown', total number of other 'unknown' fields:")
    print(total_other_unknowns.value_counts())
    print("\n")

### Preprocess Data

In [ ]:
# Reformat unknown as nan
for var in missing_vars:
    # Change unknown to nan variables to so that we can use dataframe functions to drop
    df_impute[f'{var}'] = [np.nan if x=='unknown' else x for x in df_impute[var]]
    # Create missingness indicator
    df_impute[f'{var}_missing'] = df_impute[var].isna().astype(int)

## Missingness Tests

### Little's MCAR test
test the null hypothesis that data is MCAR. Significant results suggest data is not MCAR

In [ ]:
for var in missing_vars:
    # Adjust predictors
    adjusted_predictors = [val for val in predictors if val!=var]
    categorical_predictors = [val for val in predictors if val!='age_firstLBP' and val!='startdatekey_firstLBP']
    print(var)
    print(adjusted_predictors)
    print(categorical_predictors)
    # Drop rows where the 'adjusted_predictors' columns have missing values
    df2 = df_impute.dropna(subset=adjusted_predictors)  # Drop rows with missing predictors
    
    df_summary = TableOne(df2, 
                          groupby=f'{var}_missing', 
                          categorical=categorical_predictors,
                          pval=True)
    print(df_summary)

### VIF

In [ ]:
# Variation inflation factor (VIF)

"""
Variance Inflation Factor (VIF) is a metric used to detect multicollinearity in a regression model. Multicollinearity occurs 
when predictor variables are highly correlated, which can make it difficult to estimate the effect of each predictor accurately
due to inflate standard errors of coefficients, reduced statistical power, and unreliability regression estimates.

Interpreting VIF:
* VIF=1: No correlation with other variables.
* 1<VIF<5: Moderate correlation; generally acceptable.
* VIF≥5: High multicollinearity; consider addressing it.
* VIF≥10: Severe multicollinearity; problematic.
"""

for var in missing_vars:
    # Adjust predictors
    adjusted_predictors = [val for val in predictors if val!=var]
    categorical_predictors = [val for val in adjusted_predictors if val!='age_firstLBP' and val!='startdatekey_firstLBP']
    
    # Drop rows where the 'adjusted_predictors' columns have missing values
    df2 = df_impute.dropna(subset=adjusted_predictors)  # Drop rows with missing predictors
    
    y = df2[f'{var}_missing'].to_numpy()
    X = df2[adjusted_predictors]
    
    # Encode the input variable and outcome variable as numbers
    for cat_var in categorical_predictors:
        label_encoder = LabelEncoder()
        X[cat_var] = label_encoder.fit_transform(X[cat_var].astype('str'))
    
    # Normalize each column/variable
    X = (X-X.min())/(X.max()-X.min())
    
    # Add a constant for the intercept
    X = sm.add_constant(X)
    
    # Calculate VIF for each feature
    vif_data = pd.DataFrame()
    vif_data["Variable"] = X.columns
    vif_data["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]

    print(var)
    print(vif_data)


### MAR Test with Logistic Regression

Create binary indicators (missing = 1, non-missing = 0) for each variable.
Run logistic regression with observed variables as predictors. If missingness is significantly associated with observed variables, the data is likely MAR.

Hypotheses
* Patients who are smokers may refuse to answer (MNAR).
* Missingness may depend on observed variables like age or language (MAR).
* Patients refusing to disclose religion may be systematic (MNAR).
* Missingness could relate to observed variables like sex or age (MAR).
* Non-response might depend on language or demographic reluctance (MAR).

Interpretation
Logistic Regression Coefficients
* Coefficiens: Indicates the direction and strength of the relationship between a predictor and the outcome (missingness).
    * Positive (+): Increases the likelihood of the event (missing data).
    * Negative (−): Decreases the likelihood of the event (missing data).
* P-value: Shows the statistical significance of the predictor.
    * 𝑝<0.05: Statistically significant.
    * 𝑝≥0.05: Not statistically significant.
* Meaning:
    * +β, p<0.05 = The predictor increases the likelihood of missingness.
    * -β, p<0.05 = The predictor decreases the likelihood of missingness.
    * +β, p>0.05 = The predictor has a weak or no association with missingness (trend not significant).
    * -β, p>0.05 = The predictor has a weak or no association with missingness (trend not significant).
    * B~=0 = The predictor has almost no impact on missingness.
    
Metrics like Accuracy, Precision, and Recall from the classification_report give insight into how well the model predicts missingness.
* Good model performance: Supports MAR, as the missingness is explainable by observed data.
* Poor model performance: Suggests missingness may be completely random (MCAR) or dependent on unobserved factors (MNAR).


In [ ]:
# PREDICTING MISSINGNESS WITH LOGISTIC REGRESSION
    
for var in missing_vars:
    # Adjust predictors
    adjusted_predictors = [val for val in predictors if val!=var]
    categorical_predictors = [val for val in adjusted_predictors if val!='age_firstLBP' and val!='startdatekey_firstLBP']
    
    # Drop rows where the 'adjusted_predictors' columns have missing values
    df2 = df_impute.dropna(subset=adjusted_predictors)  # Drop rows with missing predictors
    
    y = df2[f'{var}_missing'].to_numpy()
    X = df2[adjusted_predictors]
    
    # Encode the input variable and outcome variable as numbers
    for cat_var in categorical_predictors:
        X[cat_var] = label_encoder.fit_transform(X[cat_var].astype('str'))
    
    # Normalize each column/variable
    X = (X-X.min())/(X.max()-X.min())
    
    # Add a constant for the intercept
    X = sm.add_constant(X)

    # Split the dataset into training and testing sets
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # Fit the logistic regression model
    logit_model = sm.Logit(y_train, X_train)
    result = logit_model.fit()

    # Extract coefficients and p-values
    summary_table = pd.DataFrame({
        "Predictor": X.columns,
        "Coefficient": result.params.values,
        "P-value": result.pvalues.values
    })

    # Format table for better readability
    summary_table['P-value'] = summary_table['P-value'].apply(lambda p: f"{p:.3f}")
    summary_table['Coefficient'] = summary_table['Coefficient'].apply(lambda c: f"{c:.3f}")
    
    print(f"Missingness Analysis for {var}:")
    print(summary_table)

    # Predict probabilities on the test set
    y_pred_prob = result.predict(X_test)
    # Convert probabilities to binary predictions using a threshold (default = 0.5)
    y_pred = (y_pred_prob >= 0.5).astype(int)

    print(classification_report(y_test, y_pred))
    print(f"Balanced Accuracy: {balanced_accuracy_score(y_test, y_pred, adjusted=False):.2f}")
    print(f"Balanced Accuracy adjusted for chance: {balanced_accuracy_score(y_test, y_pred, adjusted=True):.2f}")
        
    # Generate the confusion matrix
    #cm = confusion_matrix(y_test, y_pred)
    #print("Confusion Matrix:\n", cm)

## Demographics Imputation for analysis
KNNImpute as the best method to fill missing values

In [ ]:
# Adjust predictors
categorical_predictors = [val for val in predictors if val!='age_firstLBP' and val!='startdatekey_firstLBP']

X = df_impute[predictors].copy()

# Encode the input variable and preserve NaN
label_encoders = {}
for cat_var in categorical_predictors:
    print(f"Encoding: {cat_var}")
    label_encoders[cat_var] = LabelEncoder()
    # Fit on non-missing values and transform while preserving NaN
    non_nan_mask = ~X[cat_var].isna()
    X.loc[non_nan_mask, cat_var] = label_encoders[cat_var].fit_transform(X.loc[non_nan_mask, cat_var].astype('str'))

X

In [ ]:
# KNN Imputation
'''
This class fills in missing values using the k-Nearest Neighbors algorithm. It estimates the missing value by averaging 
the values of the k-nearest neighbors in the feature space.
'''
imputer = KNNImputer(n_neighbors=3)
imputed_data = imputer.fit_transform(X)
X_imputed = pd.DataFrame(imputed_data, columns=X.columns)
X_imputed

In [ ]:
# Inverse transform encoded categorical variables
df_imputed = X.copy()
for cat_var in categorical_predictors:
    print(f"Decoding: {cat_var}")
    # Decode only valid encoded values, leave NaN as is
    df_imputed[cat_var] = X_imputed[cat_var].round().astype('int')  # Ensure int type for inverse transform
    valid_values_mask = (df_impute[cat_var] != X_imputed[cat_var].isna())  # Detect where there is "not" missing values
    df_imputed.loc[valid_values_mask, cat_var] = label_encoders[cat_var].inverse_transform(df_imputed.loc[valid_values_mask, cat_var].astype('int'))

df_imputed['patientdurablekey'] = merged_df['patientdurablekey'].copy()
df_imputed.head()

In [ ]:
# --- SAVE IMPUTED VALUES & ALL DATA WITH IMPUTED VALUES ---
# Save imputed data patient demographics only
df_imputed.to_csv('imputed_demographics_analysis.csv')


# All data with imputed values 
merged_df_imputed = merged_df.copy()
# Add imputed values
for var in missing_vars:
    print(f"Updating missing: {var}")
    merged_df_imputed[var] = df_imputed[var]
# Fill NaN values with 0
merged_df_imputed = merged_df_imputed.fillna(0)

# Save imputed data entire patient profile
merged_df_imputed.to_csv('imputed_patient_profile_analysis.csv')

merged_df_imputed